### 🧹 步驟 1-2：虛空幽靈防爆大掃除
* 強行校正本尊定位回到 `/content` 大廳，避免檔案系統打結。
* 清空上一場車禍的殘留檔案。

In [ ]:
# 全域設定：整個 pipeline 共用的路徑／實驗名稱，之後要換實驗只改這裡
PROJECT_DIR = "/content/ICME26-ATTM-GC-FluxAudio"
EVAL_DIR = "/content/ICME26-ATTM-GC-Evaluation"
EXP_ID = "mini_test_1k"  # 之後正式訓練時把這裡換成新的實驗名稱就好
OUTPUT_DIR = f"{PROJECT_DIR}/output/{EXP_ID}"
REF_DIR = "/content/eval_reference"
DRIVE_CHECKPOINT_DIR = f"/content/drive/MyDrive/FluxAudio_checkpoints/{EXP_ID}"


In [ ]:
# 掛載 Google Drive（訓練 checkpoint 會自動同步過去，避免斷線遺失）
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 先看看我們手上有多少食材（快取檢查：Runtime 還留著上次解壓的資料就不用重跑下載）
import os

audio_root = f"{PROJECT_DIR}/data/jamendo/mtg_jamendo_separated"

if os.path.exists(audio_root):
    folders = sorted(os.listdir(audio_root))
    print(f"有哪些資料夾: {folders}")

    total_files = 0
    for folder in folders:
        folder_path = os.path.join(audio_root, folder)
        if os.path.isdir(folder_path):
            count = len([f for f in os.listdir(folder_path) if f.endswith('_instrumental.mp3')])
            print(f"  {folder}/ → {count} 首歌")
            total_files += count

    print(f"\n總共: {total_files} 首歌")
else:
    print("目前沒有快取資料，需要往下執行下載/解壓流程（第三、四階段）")


# 🎵 ICME 2026 Grand Challenge: Text-to-Music (ATTM)
> **大會 Baseline 模型實作**：基於 Diffusion / Flow Matching 架構的音樂生成系統

---
## 🛠️ 第一階段：雲端環境初始化與套件安裝
本章節負責清理 Google Colab 雲端廚房的環境，並安裝台大音樂多媒體實驗室專案所需的現代化套件依賴。



In [ ]:
# 安裝套件
!git clone https://github.com/ntu-musicailab/ICME26-ATTM-GC-FluxAudio.git

In [ ]:
%cd {PROJECT_DIR}


In [ ]:
!pip install .

In [ ]:
# !ls -la

## 🗃️ 第二階段：輔助組件權重（Weights）下載
> ⚠️ **大會核心規範備忘錄**：
> 本競賽要求核心生成模型必須**從頭訓練（Training from scratch）**，不能使用現成的預訓練核心。
> 但為了節省算力，大會允許直接使用公開的**預訓練輔助組件**：包含負責文字語義的 **Text Encoder (T5)**，以及負責音訊極致壓縮的 **Audio Autoencoder (VAE)**。

In [ ]:
import os
from huggingface_hub import snapshot_download

# 1. 在台大專案底下，自動建立一個叫做 weights 的冰箱
os.makedirs("weights", exist_ok=True)

# 2. 啟動官方快遞，去抓 MeanAudio 的半成品高湯權重
snapshot_download(
    repo_id="AndreasXi/MeanAudio",
    local_dir="./weights"
)

In [ ]:
# 【可選】快取捷徑：跟正式版共用同一份特徵資料快取
# mini_test 跟正式版處理的是同一套完整資料集、同樣的切分參數（val=100, test=100, seed=42），
# 所以正式版存在 Drive 上的特徵快取，這裡可以直接拿來用，不用重新做一次三、四、五階段。
#
# ⚠️ 這格是「可選」的：如果你的目的是驗證第三、四、五階段的程式碼本身有沒有壞（真正的煙霧測試），
#    請不要執行這格，直接跳到下面的第三階段照原本流程跑一次完整版。
import os, subprocess

DRIVE_CACHE_ZIP = '/content/drive/MyDrive/FluxAudio_data/jamendo_meanaudio_ready_cache.zip'

if os.path.exists(DRIVE_CACHE_ZIP):
    size_gb = os.path.getsize(DRIVE_CACHE_ZIP) / 1e9
    print(f'✅ 在 Drive 找到特徵資料快取（{size_gb:.1f} GB，與正式版共用），還原中...')
    subprocess.run(['unzip', '-q', '-o', DRIVE_CACHE_ZIP, '-d', PROJECT_DIR])
    print('✅ 還原完成！可以直接跳到「第六階段：模型訓練」，不用執行第三、四、五階段')
else:
    print('Drive 上還沒有快取，請依序執行第三、四、五階段')


## 🥗 第三階段：雲端快取流 —— 使用學長預處理半成品

> 📌 **戰略備忘錄**：
> 學長已在實驗室 Server 上跑完「30 秒裁切 + Mel-Band Roformer 去人聲」，
> 成品切成 10 包 zip（00-09 到 90-99）放在 Google Drive 上。
>
> ⚠️ **已知限制**：Google Drive 公開檔案有流量限制（Quota），
> 同時間太多人下載會被擋。腳本會嘗試最多 5 次，下載不了的先跳過，
> 用手上有的資料先把 pipeline 跑通（Mini Train），剩下的之後再補。


In [ ]:
%%writefile download_clean.py
import os
import time
import subprocess

file_ids = {
    "00-09": "1t0969ijt5MOYXp56Ur5dDC5u7U4Gw_lw",
    "10-19": "1vF-Ny3PC6r-mvVBM_gT23MsZY0vWup6n",
    "20-29": "1Q-uFpTD8VzVBhvkasus8WsolTq0zQM62",
    "30-39": "1ELRp8Pu_QZdmGlumoRfYQ6aCS4FP-hmG",
    "40-49": "1LD9M1CFTJXoRNvfKyEhXdvw_Z9w-4iDz",
    "50-59": "1h2w-l0XgTK0iXayJ5SqQmlLfEQW6l8Bu",
    "60-69": "1-jtiRC32ihbGiYjqTwdOOzs4kJzKi64D",
    "70-79": "162QFwg3gu9Ks3XrDiS1RrNOPL925ZxQb",
    "80-89": "1mvU7SXvAMVIk8nH1L_8xTdjNVCxO86SW",
    "90-99": "1sjTkm2CrBDpZvyTCkxvH3V0d4XVKtbn6"
}

MAX_RETRIES = 5
target_dir = "data/jamendo/mtg_subset_30s_separated_zips"
os.makedirs(target_dir, exist_ok=True)

success_list = []
failed_list = []

for name, fid in file_ids.items():
    output_path = f"{target_dir}/mtg_full_separated_{name}.zip"

    # 已存在且大小正常就跳過
    if os.path.exists(output_path) and os.path.getsize(output_path) > 100 * 1024 * 1024:
        print(f"✅ {name}.zip 已存在，跳過")
        success_list.append(name)
        continue

    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        print(f"🔄 下載 {name}.zip（第 {attempt}/{MAX_RETRIES} 次）...")
        cmd = f'gdown "https://drive.google.com/uc?id={fid}" -O {output_path} --fuzzy --remaining-ok'
        result = subprocess.run(cmd, shell=True)

        if result.returncode == 0 and os.path.exists(output_path) and os.path.getsize(output_path) > 100 * 1024 * 1024:
            print(f"🎉 {name}.zip 下載成功！")
            success_list.append(name)
            success = True
            break
        else:
            if os.path.exists(output_path):
                os.remove(output_path)
            if attempt < MAX_RETRIES:
                print(f"⏳ 被 Google 限流，等 10 秒後重試...")
                time.sleep(10)

    if not success:
        print(f"⛔ {name}.zip 下載失敗，已達重試上限，先跳過")
        failed_list.append(name)

print("\n" + "=" * 50)
print(f"✅ 成功下載: {success_list}")
print(f"⛔ 未能下載: {failed_list}")
print(f"📊 成功 {len(success_list)}/{len(file_ids)} 包")
print("=" * 50)


In [ ]:
%cd {PROJECT_DIR}

# 執行下載（最多重試 5 次就會跳過）
!python3 download_clean.py

# 解壓所有成功下載的 zip
import glob
import subprocess

zip_dir = "data/jamendo/mtg_subset_30s_separated_zips"
out_dir = "data/jamendo/mtg_jamendo_separated"
!mkdir -p {out_dir}

zips = sorted(glob.glob(f"{zip_dir}/*.zip"))
print(f"\n找到 {len(zips)} 個 zip 檔，開始解壓...")

for z in zips:
    print(f"📦 解壓 {os.path.basename(z)}...")
    subprocess.run(["unzip", "-q", "-o", z, "-d", out_dir])

# 驗證結果
import os
folders = sorted([f for f in os.listdir(out_dir) if os.path.isdir(os.path.join(out_dir, f))])
total = sum(len([x for x in os.listdir(os.path.join(out_dir, f)) if x.endswith('.mp3')]) for f in folders)
print(f"\n✅ 解壓完成！共 {len(folders)} 個資料夾，{total} 首歌")
print(f"📁 資料夾: {folders}")


## 📐 第四階段：資料前處理 —— 切分 Train / Val / Test

將 55,701 首已去人聲的樂器伴奏，與大會提供的文字描述（caption）做比對，
篩選出「同時有音檔 + 有文字描述」的有效樣本，
再隨機切成訓練集、驗證集、測試集三份。


In [ ]:
%cd {PROJECT_DIR}

!python training/prepare_jamendo_for_meanaudio.py \
    --audio_root ./data/jamendo/mtg_jamendo_separated \
    --caption_path ./data/captions/jamendo_qwen.json \
    --output_dir ./data/jamendo_meanaudio_ready \
    --val_samples 100 \
    --test_samples 100


In [ ]:
# 確認各 split 的樣本數
!echo "=== Train ===" && wc -l ./data/jamendo_meanaudio_ready/train/jamendo_train.tsv
!echo "=== Val ===" && wc -l ./data/jamendo_meanaudio_ready/val/jamendo_val.tsv
!echo "=== Test ===" && wc -l ./data/jamendo_meanaudio_ready/test/jamendo_test.tsv

# 看看 TSV 長什麼樣子（前 3 筆）
!echo "\n=== Train 前 3 筆 ===" && head -3 ./data/jamendo_meanaudio_ready/train/jamendo_train.tsv


## 🧬 第五階段：特徵萃取（Feature Extraction）

將每首 30 秒音檔切成 10 秒片段，再透過預訓練的輔助組件：
- **VAE（Audio Autoencoder）**：將音訊壓縮成低維度的 latent 向量
- **T5 + CLAP（Text Encoder）**：將文字描述編碼成語義向量

最終輸出 `.npz` 檔，供後續模型訓練直接使用。

> ⚠️ 這個步驟需要 GPU，且資料量大（55K 首歌），預計需要較長時間。


In [ ]:
%cd {PROJECT_DIR}

# === 步驟 1：切割音檔成 10 秒片段 ===
!python training/partition_clips.py \
    --data_dir ./data/jamendo_meanaudio_ready/train/audios \
    --output_dir ./data/jamendo_meanaudio_ready/train/partitions.tsv

# === 步驟 2：萃取 VAE latent + T5 text embedding ===
!torchrun --standalone --nproc_per_node=1 training/extract_audio_latents.py \
    --captions_tsv ./data/jamendo_meanaudio_ready/train/jamendo_train.tsv \
    --data_dir ./data/jamendo_meanaudio_ready/train/audios \
    --clips_tsv ./data/jamendo_meanaudio_ready/train/partitions.tsv \
    --latent_dir ./data/jamendo_meanaudio_ready/train/latents \
    --output_dir ./data/jamendo_meanaudio_ready/train/npz \
    --text_encoder='t5_clap'


In [ ]:
# === Val split ===
!python training/partition_clips.py \
    --data_dir ./data/jamendo_meanaudio_ready/val/audios \
    --output_dir ./data/jamendo_meanaudio_ready/val/partitions.tsv

!torchrun --standalone --nproc_per_node=1 training/extract_audio_latents.py \
    --captions_tsv ./data/jamendo_meanaudio_ready/val/jamendo_val.tsv \
    --data_dir ./data/jamendo_meanaudio_ready/val/audios \
    --clips_tsv ./data/jamendo_meanaudio_ready/val/partitions.tsv \
    --latent_dir ./data/jamendo_meanaudio_ready/val/latents \
    --output_dir ./data/jamendo_meanaudio_ready/val/npz \
    --text_encoder='t5_clap'


In [ ]:
# === Test split ===
!python training/partition_clips.py \
    --data_dir ./data/jamendo_meanaudio_ready/test/audios \
    --output_dir ./data/jamendo_meanaudio_ready/test/partitions.tsv

!torchrun --standalone --nproc_per_node=1 training/extract_audio_latents.py \
    --captions_tsv ./data/jamendo_meanaudio_ready/test/jamendo_test.tsv \
    --data_dir ./data/jamendo_meanaudio_ready/test/audios \
    --clips_tsv ./data/jamendo_meanaudio_ready/test/partitions.tsv \
    --latent_dir ./data/jamendo_meanaudio_ready/test/latents \
    --output_dir ./data/jamendo_meanaudio_ready/test/npz \
    --text_encoder='t5_clap'


In [ ]:
import os

for split in ['train', 'val', 'test']:
    npz_dir = f"./data/jamendo_meanaudio_ready/{split}/npz"
    if os.path.exists(npz_dir):
        count = len([f for f in os.listdir(npz_dir) if f.endswith('.npz')])
        print(f"{split}: {count} 個 .npz 檔")
    else:
        print(f"{split}: npz 資料夾不存在")


## 🚀 第六階段：模型訓練（Training FluxAudio-S）

使用已萃取的 .npz 特徵檔，從零開始訓練 FluxAudio-S（120M 參數）。
訓練過程透過 Weights & Biases 即時監控 loss 曲線。

> ⚠️ 完整訓練需要 200K iterations，時間很長。
> 建議先跑少量 iterations 確認 pipeline 正常，再決定要跑多久。


In [ ]:
# 設定 W&B
# Weights & Biases 登入（第一次需要貼上 API key）
# 去 https://wandb.ai/authorize 取得你的 key
!pip install wandb -q
import wandb
wandb.login()


In [ ]:
# 安裝 av-benchmark 評估工具
%cd /content

# 克隆 av-benchmark 評估工具
!git clone https://github.com/hkchengrex/av-benchmark.git

# 安裝它的依賴
%cd /content/av-benchmark
!pip install -e . -q

# 建立 symlink 讓 FluxAudio 找得到 av_bench
%cd {PROJECT_DIR}
!ln -sf /content/av-benchmark/av_bench ./av_bench

# 驗證 symlink 是否正確
!ls -la av_bench



In [ ]:
# 設定 checkpoint 自動同步到 Google Drive
# train.py 會把權重存到 exps/{EXP_ID}/，這裡用 symlink 讓這個路徑實際指向 Drive，
# 這樣訓練中每次存檔都直接寫進 Drive，不用等訓練結束才手動下載，Colab 斷線也不會遺失。
%cd {PROJECT_DIR}
import os

os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs("exps", exist_ok=True)

local_exp_dir = f"exps/{EXP_ID}"
if os.path.islink(local_exp_dir):
    print(f"{local_exp_dir} 已經是 symlink，略過")
elif os.path.exists(local_exp_dir):
    print(f"⚠️ {local_exp_dir} 已存在且不是 symlink，請確認後手動處理，本次不覆蓋")
else:
    os.symlink(DRIVE_CHECKPOINT_DIR, local_exp_dir)
    print(f"已建立 symlink: {local_exp_dir} -> {DRIVE_CHECKPOINT_DIR}")


In [ ]:
# 開始訓練（先跑 1000 步測試）

%cd {PROJECT_DIR}

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 先跑 1000 步確認一切正常，之後再加大
!torchrun --standalone --nproc_per_node=1 \
    train.py \
    --config-name train_config_jamendo.yaml \
    exp_id={EXP_ID} \
    compile=False \
    model=fluxaudio_s \
    batch_size=32 \
    eval_batch_size=4 \
    num_iterations=1000 \
    text_encoder_name=t5_clap \
    data_dim.text_c_dim=512 \
    pin_memory=False \
    num_workers=4 \
    ac_oversample_rate=5 \
    use_meanflow=False \
    cfg_strength=4.5 \
    ++use_rope=True \
    ++use_wandb=True \
    ++debug=False \
    val_interval=500 \
    save_checkpoint_interval=500 \
    log_text_interval=50


In [ ]:
# 刪掉已經解壓完畢、不再需要的 zip 檔，釋放約 27GB 空間
import shutil

zip_dir = f"{PROJECT_DIR}/data/jamendo/mtg_subset_30s_separated_zips"
shutil.rmtree(zip_dir, ignore_errors=True)

!df -h /content


## 🎵 第七階段：推論（Inference）—— 用模型生成音樂

用訓練好的 checkpoint，輸入文字描述，讓模型生成對應的音樂。

> ⚠️ 只訓練了 1000 步，生成的音樂品質一定很差（可能像雜訊），
> 這是正常的。目的是確認整個 pipeline 能跑通。


In [ ]:
# 生成音樂

%cd {PROJECT_DIR}

# 直接用訓練存的權重，不需要 unwrap
prompts = [
    "A calm acoustic guitar melody with soft percussion",
    "An energetic electronic dance track with heavy bass",
    "A peaceful piano piece with ambient strings",
]

for i, prompt in enumerate(prompts):
    !python infer.py \
        --variant fluxaudio_s \
        --model_path exps/{EXP_ID}/{EXP_ID}_last.pth \
        --encoder_name t5_clap \
        --use_rope \
        --prompt "{prompt}" \
        --output {OUTPUT_DIR} \
        --seed {i + 42}

!ls -la {OUTPUT_DIR}/


In [ ]:
# 在 Colab 裡試聽生成的音樂

import IPython.display as ipd
import glob

wav_files = sorted(glob.glob(f"{OUTPUT_DIR}/*.wav"))
for wav in wav_files:
    print(f"\n🎵 {wav.split('/')[-1]}")
    display(ipd.Audio(wav))


In [ ]:
# 將 wav files 存到本地端

# 先把 3 個 wav 打包成一個 zip，一次下載
!zip -j /content/{EXP_ID}_audio.zip {OUTPUT_DIR}/*.wav

from google.colab import files
files.download(f"/content/{EXP_ID}_audio.zip")


## 📊 第八階段：評估（Evaluation）

使用兩個指標衡量生成音樂的品質：
- **FAD（Frechet Audio Distance）**：數字越低越好，代表生成的音樂與真實音樂的分佈越接近
- **CLAP（Text-Audio Similarity）**：數字越高越好，代表生成的音樂與文字描述越匹配

> ⚠️ 只訓練 1000 步，分數一定很差，這是正常的。
> 目的是確認評估 pipeline 能跑通。


In [ ]:
# 安裝評估工具

%cd /content

# 克隆評估工具（如果已存在就跳過）
import os
if not os.path.exists(EVAL_DIR):
    !git clone https://github.com/ntu-musicailab/ICME26-ATTM-GC-Evaluation.git {EVAL_DIR}

%cd {EVAL_DIR}
!pip install -r requirements.txt -q

# 用 FluxAudio 已有的完好 checkpoint 覆蓋評估工具的
import shutil
os.makedirs(f"{EVAL_DIR}/load/clap_score", exist_ok=True)
shutil.copy2(
    f"{PROJECT_DIR}/weights/music_speech_audioset_epoch_15_esc_89.98.pt",
    f"{EVAL_DIR}/load/clap_score/music_speech_audioset_epoch_15_esc_89.98.pt",
)

# 修復 laion_clap 的 logging bug
import site
clap_hook = site.getsitepackages()[0] + "/laion_clap/hook.py"
with open(clap_hook, "r") as f:
    content = f.read()
patched = content.replace(
    'logging.info(n, "\\t", "Loaded" if n in ckpt else "Unloaded")',
    'logging.info(f"{n}\\tLoaded" if n in ckpt else f"{n}\\tUnloaded")',
)
if patched == content:
    print("⚠️ 沒有找到要替換的字串，laion_clap 版本可能已經變了，請手動檢查")
else:
    with open(clap_hook, "w") as f:
        f.write(patched)
    print("評估工具安裝完成，checkpoint 已就位，logging bug 已修復")


In [ ]:
# 準備 ground truth 音檔供 FAD 比較

# FAD 需要比較「真實音樂」和「生成音樂」的差距
# 從 test set 複製一些真實音檔出來當 reference
import shutil, os

os.makedirs(REF_DIR, exist_ok=True)

test_audio_dir = f"{PROJECT_DIR}/data/jamendo_meanaudio_ready/test/audios"
test_files = [f for f in os.listdir(test_audio_dir) if f.endswith(".mp3")]

for f in test_files:
    shutil.copy2(os.path.join(test_audio_dir, f), REF_DIR)

print(f"Reference 音檔: {len(os.listdir(REF_DIR))} 首")


In [ ]:
# 計算 FAD（Fréchet Audio Distance）
# 跟正式版用同一支腳本、同一種呼叫方式：直接用 !python 執行，
# 這樣如果失敗，錯誤訊息會直接印在 Colab 畫面上，不會被吞掉。
%cd {EVAL_DIR}

!python src/fad.py {REF_DIR} {OUTPUT_DIR} -w 1


In [ ]:
# 準備 CLAP 評估用的 CSV

import csv

# CLAP 需要一個 CSV 檔，記錄每首生成音樂對應的文字描述
clap_csv = "/content/clap_eval.csv"
entries = [
    ("A_calm_acoustic_guitar_melody_with_soft_percussion--numsteps25--seed42", "A calm acoustic guitar melody with soft percussion"),
    ("An_energetic_electronic_dance_track_with_heavy_bass--numsteps25--seed43", "An energetic electronic dance track with heavy bass"),
    ("A_peaceful_piano_piece_with_ambient_strings--numsteps25--seed44", "A peaceful piano piece with ambient strings"),
]

with open(clap_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "caption"])
    for audio_id, caption in entries:
        writer.writerow([audio_id, caption])

print("CLAP CSV 已建立")
!cat /content/clap_eval.csv


In [ ]:
# 計算 CLAP

%cd {EVAL_DIR}

# 參數順序：先 CSV，再音檔資料夾
!python src/clap.py \
    /content/clap_eval.csv \
    {OUTPUT_DIR}


In [ ]:
# 讀取 FAD / CLAP 分數

import csv, os

print("=== FAD Score ===")
for path in [f"{EVAL_DIR}/result/fad.csv", "./result/fad.csv"]:
    if os.path.exists(path):
        with open(path) as f:
            for row in csv.reader(f):
                print(row)
        break
else:
    print("FAD 結果找不到")

print("\n=== CLAP Score ===")
for path in [f"{EVAL_DIR}/result/clap.csv", "./result/clap.csv"]:
    if os.path.exists(path):
        with open(path) as f:
            for row in csv.reader(f):
                print(row)
        break
else:
    print("CLAP 結果找不到")


In [ ]:
# 打包 checkpoint 存到本地端

from google.colab import files
!zip -j /content/checkpoint_{EXP_ID}.zip \
    {PROJECT_DIR}/exps/{EXP_ID}/{EXP_ID}_last.pth \
    {PROJECT_DIR}/exps/{EXP_ID}/{EXP_ID}_ckpt_last.pth
files.download(f"/content/checkpoint_{EXP_ID}.zip")
